# Data Augmentation: Perturb vs Generate

Notebooks 1-6 cover record, stream, train, deploy, and fleet orchestration.
This one covers **data augmentation**: three mechanisms ship today, and they
act at three different stages of that pipeline.

| # | Stage | Mechanism | What it diversifies |
|---|-------|-----------|---------------------|
| 1 | Collection time | `randomize()` + `set_obs_noise()` | appearance, physics, and sensor noise *while episodes are being produced* |
| 2 | Render time | backdrop compositing (`HybridCompositor.set_background`) | the visual domain over the *same* trajectories |
| 3 | Train time | `TrainSpec.augmentation` | classical image augmentation inside the training container (GR00T-provider-specific) |

All three **perturb**: they diversify *within* the simulator's visual and
dynamic distribution. The fourth mechanism - **generative** augmentation,
which escapes that distribution - ships as `strands_robots.transforms`, but
generation runs a video2video model that needs a GPU, and this notebook is
CPU-only - so section 4 documents it in markdown only
([`docs/data/transforms.md`](https://github.com/strands-labs/robots/blob/main/docs/data/transforms.md)
is the full reference).

Everything below runs end-to-end on CPU: no hardware, no GPU, no Hugging Face
credentials.

**Requirements:**

```bash
pip install -U "strands-robots[sim-mujoco,lerobot]"
```

Keep the `-U` (or a version floor) on any install line you adapt: without it,
an environment that already carries an older release reports
`Requirement already satisfied` and runs the notebook against the stale one.

In [ ]:
import os
import shutil
import sys

# macOS uses "cgl" for offscreen GL; Linux headless uses "egl".
os.environ.setdefault("MUJOCO_GL", "cgl" if sys.platform == "darwin" else "egl")
os.environ.setdefault("STRANDS_TRUST_REMOTE_CODE", "1")

SEED = 0  # every seeded call below takes this, so a re-run reproduces the run
ROOT = "/tmp/nb7_dataset"

shutil.rmtree(ROOT, ignore_errors=True)


def check(result, what):
    """Raise if a sim action returned an error dict - never continue silently."""
    if isinstance(result, dict) and result.get("status") == "error":
        msg = "; ".join(c.get("text", "") for c in result.get("content", []))
        raise RuntimeError(f"{what} failed: {msg}")
    return result

## 1. Collection time: randomize the world you record

`randomize()` perturbs the scene itself - geom colors, light position and
intensity, per-geom friction, per-body mass - and `set_obs_noise()` adds
sensor noise (Gaussian joint noise, camera pixel jitter) to every observation
that follows. Both take a `seed`, so a run is reproducible. Applied while
recording, the perturbations land *in the dataset*: a policy trained on it
sees many scenes instead of one pristine scene, which is the sim2real point.

This is the machinery of
[`examples/12_domain_randomization.py`](https://github.com/strands-labs/robots/blob/main/examples/12_domain_randomization.py).
First, a pristine baseline: a SO-100 arm, a red cube, and a camera.

In [ ]:
from PIL import Image

from strands_robots import MockPolicy, Robot

sim = Robot("so100", mesh=False)
check(
    sim.add_object(
        name="cube",
        shape="box",
        size=[0.025, 0.025, 0.025],
        position=[0.2, 0.0, 0.05],
        color=[1, 0, 0, 1],
        mass=0.05,
    ),
    "add_object",
)
check(sim.add_camera(name="view", position=[0.5, 0.0, 0.4], target=[0.2, 0.0, 0.05]), "add_camera")

baseline = sim.get_observation("so100")["view"]  # (H, W, 3) uint8 camera frame
Image.fromarray(baseline)

Now perturb it. `randomize()` returns a status dict describing exactly what
changed - including the per-geom friction and per-body mass scales, so the
physics half of the perturbation is inspectable, not just visible. The same
`seed` reproduces the same perturbation on a fresh scene.

In [ ]:
result = check(
    sim.randomize(randomize_colors=True, randomize_lighting=True, randomize_physics=True, seed=SEED),
    "randomize",
)
# The summary lines; the full per-geom/per-body scale dicts follow them in the same payload.
print("\n".join((result.get("content") or [{}])[0].get("text", "").splitlines()[:4]))
Image.fromarray(sim.get_observation("so100")["view"])

`set_obs_noise()` is the sensor half: real encoders and cameras are noisy, and
a policy trained on noiseless readings overfits to precision that hardware
does not have. Once set, the noise applies to **every** observation until
reconfigured (all-zero stds disable it exactly).

In [ ]:
obs = sim.get_observation("so100")
# A scalar joint reading: any float key that is not a velocity or a base-pose field.
joint_key = next(
    k for k, v in obs.items() if not k.endswith(".vel") and not k.startswith("base") and isinstance(v, float)
)
clean = float(obs[joint_key])
check(sim.set_obs_noise(joint_pos_std=0.02, joint_vel_std=0.05, camera_jitter_px=1.0, seed=SEED), "set_obs_noise")
noisy = float(sim.get_observation("so100")[joint_key])
print(f"joint {joint_key!r}: clean={clean:.4f} rad  noisy={noisy:.4f} rad  (delta={noisy - clean:+.4f})")

Record an episode with both applied. The recorder captures what the policy
observes, so the randomized appearance, the perturbed physics, and the
injected sensor noise all land in the dataset. As in notebooks 2-5:
`control_frequency` must match the recording `fps` - the recorder writes one
frame per control step with no decimation, so a mismatched rate is refused
rather than written at distorted timestamps.

In [ ]:
check(
    sim.start_recording(
        repo_id="local/nb7_demo",
        root=ROOT,
        fps=30,
        task="pick up the red cube",
        cameras=["view"],  # record only the declared sensor, not the implicit 'default' view
        overwrite=True,
    ),
    "start_recording",
)
rollout = sim.run_policy(
    robot_name="so100",
    policy_object=MockPolicy(),
    instruction="pick up the red cube",
    n_steps=60,
    control_frequency=30.0,
)
if rollout.get("status") != "success":
    raise RuntimeError(f"rollout failed: {rollout}")
check(sim.stop_recording(), "stop_recording")

reader = sim.stream_dataset("local/nb7_demo", root=ROOT, shuffle=False, max_num_shards=1, buffer_size=1)
sim.destroy()
print(f"recorded: episodes={reader.num_episodes} frames={reader.num_frames} -> {ROOT}")

In a real collection run you re-`randomize()` between episodes (a fresh seed
per episode) so every episode samples a different world - that loop is
`examples/12_domain_randomization.py` plus the recording cell above.

## 2. Render time: swap the backdrop over the same trajectories

The second mechanism does not touch the trajectories at all.
`HybridCompositor` renders the simulation's foreground (with its depth
buffer), renders a background from a separate renderer, and composites the two
per pixel by depth - so one recorded episode can be re-rendered over any
number of visual domains.

Two background renderers ship:

- **`PanoramaBackground`** - an equirectangular panorama at infinity. With no
  arguments it generates a procedural indoor panorama (zero assets, pure
  NumPy - the CPU path this notebook uses); pass `image_path=` for your own
  panorama.
- **`GsplatBackground`** - a photoreal 3D Gaussian Splatting scene
  (`.ply`/`.spz`). CUDA-only; covered in the note after this demo.

`set_background()` is the swap. The `change_background` tool in
[`examples/isaac_gs/`](https://github.com/strands-labs/robots/blob/main/examples/isaac_gs/)
is a thin agent-tool shim over this same call. The world is created with
`ground_plane=False` so the backdrop is visible past the robot.

In [ ]:
from strands_robots import Simulation
from strands_robots.rendering import HybridCompositor, PanoramaBackground

gs_sim = Simulation(mesh=False)
check(gs_sim.create_world(ground_plane=False), "create_world")  # let the backdrop show past the robot
check(gs_sim.add_robot("arm", data_config="so101"), "add_robot")
check(gs_sim.add_camera(name="front", position=[0.4, -0.5, 0.3], target=[0.0, 0.0, 0.1]), "add_camera")
gs_sim.step(20)

compositor = HybridCompositor(gs_sim, background=PanoramaBackground())  # procedural panorama: zero assets
frame_a = compositor.render(camera_name="front")
Image.fromarray(frame_a.rgb)

In [ ]:
compositor.set_background(PanoramaBackground(rotation_deg=120.0))  # the swap: same scene, new visual domain
frame_b = compositor.render(camera_name="front")

changed = (frame_a.background_rgb != frame_b.background_rgb).any(axis=-1).mean()
print(f"backdrop pixels changed by the swap: {changed:.0%}")
gs_sim.destroy()
Image.fromarray(frame_b.rgb)

### Photoreal backdrops need a CUDA host

`GsplatBackground` drops a photoreal 3D Gaussian Splatting scene behind the
same compositor. It is CUDA-only: on a CPU-only host it refuses with a
`RuntimeError` that names `PanoramaBackground` as the alternative - an
explicit refusal rather than a silent visual downgrade, which is the lesson of
[#2321](https://github.com/strands-labs/robots/issues/2321). On a CUDA host
the swap is the same one call:

```python
from strands_robots.rendering import GsplatBackground, download_gsplat_scene, gsplat_rasterizer_available

available, reason = gsplat_rasterizer_available()  # probes a real 1-gaussian rasterization
if available:
    ply = download_gsplat_scene("tabletop (indoor room)")  # cached under ~/.cache/strands_robots
    compositor.set_background(GsplatBackground(ply, device="cuda"))
```

See [`examples/mujoco_gs/`](https://github.com/strands-labs/robots/blob/main/examples/mujoco_gs/)
and [`examples/isaac_gs/`](https://github.com/strands-labs/robots/blob/main/examples/isaac_gs/)
for the full photoreal apps built on this.

## 3. Train time: `TrainSpec.augmentation`

The third mechanism runs inside the training container: classical image
augmentation applied to every training batch. `TrainSpec.augmentation` is a
**backend-specific** dict - the GR00T provider maps `random_rotation_angle`
and `color_jitter_params` to the fine-tune script's flags of the same name and
bundles any other keys into `--extra_augmentation_config` as JSON; other
providers define their own keys.

`Gr00tTrainer.build_command()` is a pure argv helper - it launches nothing -
so the exact flags a training run would receive are inspectable without a GPU
or a GR00T checkout. (Actually launching GR00T fine-tuning needs a GPU;
notebooks 3 and 5 run the CPU-trainable path.)

In [ ]:
from strands_robots.training import TrainSpec
from strands_robots.training.groot import Gr00tTrainer

spec = TrainSpec(
    dataset_root=ROOT,  # the randomized dataset from section 1
    base_model="nvidia/GR00T-N1.5-3B",
    output_dir="/tmp/nb7_finetune",
    embodiment="new_embodiment",
    steps=100,
    seed=SEED,
    augmentation={
        "random_rotation_angle": 10,  # degrees of random in-plane rotation
        "color_jitter_params": "0.3,0.4,0.5,0.08",  # brightness, contrast, saturation, hue
    },
)

command = Gr00tTrainer().build_command(spec)
aug_flags = [arg for arg in command if "rotation" in arg or "jitter" in arg]
assert aug_flags, f"no augmentation flags in {command}"
print("\n".join(aug_flags))

## 4. Generative augmentation (markdown-only: generation needs a GPU)

The three mechanisms above **perturb**: randomized scenes, swapped backdrops,
and jittered training batches are all draws from within the simulator's visual
and dynamic distribution. What none of them can produce is content the
simulator cannot render - rain on a lens, a cluttered real kitchen, the motion
blur of a real camera. **Generative** augmentation escapes the distribution: a
generative model (Cosmos-Transfer-style) rewrites recorded episodes into new
visual worlds while keeping the action trajectories.

That mechanism ships as `strands_robots.transforms`: a **dataset transform** -
LeRobotDataset in, LeRobotDataset out - created with
`create_transform("cosmos_transfer", ...)` and configured through a
`TransformSpec`, the data-side peer of `create_policy` / `create_trainer`. It
carries a provenance doctrine the perturbing mechanisms do not need:

- **Provenance**: every generated episode is marked `synthetic=true` in the
  output dataset's provenance record, so a training run can weight, filter, or
  audit it (`load_provenance()` / `synthetic_episode_indices()`).
- **Re-validation**: generated episodes are re-checked by the same
  deterministic predicates that scored the originals - generation can change
  what an episode *shows*, so its labels cannot be trusted transitively.
- **Discard on verdict flip**: an episode whose predicate verdict changes
  under generation is dropped, never delivered - and counted
  (`TransformResult.episodes_discarded`).

This section stays markdown-only because generation needs a GPU and this
notebook is CPU-only end to end: committing cells whose output a CPU host
cannot reproduce is how a gallery ends up showing images no user can reproduce
([#2321](https://github.com/strands-labs/robots/issues/2321) is the
precedent).
[`docs/data/transforms.md`](https://github.com/strands-labs/robots/blob/main/docs/data/transforms.md)
is the full reference - the contract, the usage, and the acceptance gate.

## Where to go from here

- [`examples/12_domain_randomization.py`](https://github.com/strands-labs/robots/blob/main/examples/12_domain_randomization.py)
  is the CLI version of section 1 - loop it with a fresh seed per episode for
  a real randomized collection run.
- [`examples/mujoco_gs/`](https://github.com/strands-labs/robots/blob/main/examples/mujoco_gs/)
  and [`examples/isaac_gs/`](https://github.com/strands-labs/robots/blob/main/examples/isaac_gs/)
  run photoreal Gaussian-splat backdrops on a CUDA host.
- [`03_record_train_deploy.ipynb`](03_record_train_deploy.ipynb) and
  [`05_streaming_data_loop.ipynb`](05_streaming_data_loop.ipynb) are the
  training loops that consume a dataset like the one recorded here.
- [`docs/data/transforms.md`](https://github.com/strands-labs/robots/blob/main/docs/data/transforms.md)
  documents the generative transform section 4 describes - run it on a GPU
  host against the dataset recorded in section 1.